In [4]:

import json
import time

import psycopg
from psycopg import OperationalError


In [5]:
def load_data(filename):
    """
    Loads cleaned data from a JSON file.
    """
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

In [6]:
directory = '/Users/jennifer/Documents/software_concept_python_class/jhu_software_concepts/Module_2/'

applicant_data= load_data(directory+ "applicant_data_v2.json")
len(applicant_data)

41000

In [7]:
print(applicant_data[0].keys()) 

dict_keys(['Program Name', 'University', 'Comments', 'Date of Information Added to Grad Cafe', 'URL link to applicant entry', 'Applicant Status', 'Accepted: Acceptance Date', 'Rejected: Rejection Date', 'Semester and Year of Program Start', 'International / American Student', 'GRE Score', 'GRE V Score', 'Masters or PhD', 'GPA', 'GRE AW', 'extra_info', 'raw_texts', 'page_number', 'llm-generated-program', 'llm-generated-university'])


In [9]:
# Create connection to system database
def create_system_connection(user, password):
    """Connect to postgres system database"""
    try:
        conn = psycopg.connect(
            dbname="postgres",
            user=user,
            password=password,
            host="localhost"
        )
        conn.autocommit = True
        return conn
    except OperationalError as e:
        print(f"Connection failed: {e}")
        return None

In [11]:
system_conn = create_system_connection("postgres", "181818")

In [12]:
# Create new database
def create_database(conn, db_name):
    """Create a new database"""
    try:
        cur = conn.cursor()
        cur.execute(f"CREATE DATABASE {db_name};")
        cur.close()
        print(f"Database '{db_name}' created successfully")
    except OperationalError as e:
        print(f"Database creation failed: {e}")

create_database(system_conn, "gradcafe_db")
system_conn.close()

Database 'gradcafe_db' created successfully


In [ ]:
def create_db_connection(db_name, db_user, db_password, db_host, db_port):

    """
    create conenction to my db
    """

    connection = None
    try:
        connection = psycopg.connect(
            dbname=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
        )
        print("Connection to PostgreSQL DB successful")
    except OperationalError as e:
        print(f"The error '{e}' occurred")
    return connection

In [23]:
connection = create_db_connection("gradcafe_db", "postgres", "181818", "127.0.0.1", "5432")

Connection to PostgreSQL DB successful


In [ ]:
def execute_query(connection, query):
    connection.autocommit = True
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        # print("Query executed successfully")
    except OperationalError as e:
        print(f"The error '{e}' occurred")

#### create table

In [26]:
create_users_table = """
CREATE TABLE IF NOT EXISTS applicants (
p_id SERIAL PRIMARY KEY,
program	TEXT,
comments TEXT,
date_added date,
url	TEXT,
status TEXT,
term TEXT,
us_or_international	TEXT,
gpa	FLOAT,
gre	FLOAT,
gre_v FLOAT,
gre_aw FLOAT,
degree TEXT,
llm_generated_program TEXT,
llm_generated_university TEXT
)
"""

execute_query(connection, create_users_table)

Query executed successfully


#### insert records

In [27]:
# check datatype of the json data file
for key, value in applicant_data[0].items():
    print(key, type(value))

Program Name <class 'str'>
University <class 'str'>
Comments <class 'NoneType'>
Date of Information Added to Grad Cafe <class 'str'>
URL link to applicant entry <class 'str'>
Applicant Status <class 'str'>
Accepted: Acceptance Date <class 'NoneType'>
Rejected: Rejection Date <class 'str'>
Semester and Year of Program Start <class 'str'>
International / American Student <class 'str'>
GRE Score <class 'NoneType'>
GRE V Score <class 'NoneType'>
Masters or PhD <class 'str'>
GPA <class 'str'>
GRE AW <class 'NoneType'>
extra_info <class 'str'>
raw_texts <class 'list'>
page_number <class 'int'>
llm-generated-program <class 'str'>
llm-generated-university <class 'str'>


In [ ]:
# sample json data
# [
#     {
#         "Program Name": "Bioinformatics Masters",
#         "University": "San Jose State University",
#         "Comments": null,
#         "Date of Information Added to Grad Cafe": "May 31, 2026",
#         "URL link to applicant entry": "https://www.thegradcafe.com/result/1020289",
#         "Applicant Status": "Rejected",
#         "Accepted: Acceptance Date": null,
#         "Rejected: Rejection Date": "May 31",
#         "Semester and Year of Program Start": "Fall 2026",
#         "International / American Student": "American",
#         "GRE Score": null,
#         "GRE V Score": null,
#         "Masters or PhD": "Masters",
#         "GPA": "3.20",
#         "GRE AW": null,

In [28]:
def format_text(value):
    """
    The current value: 
    1.if the value is None,it returns "Null"
    2. replace ' with '' (SQL needs it)
    3. the whole thing quoted with ' ' (SQL needs it)
    """
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def format_float(value):
    """
    1.if the value is None,it returns "Null"
    2. extract float from json, then convert to string for SQL insertion
    """
    if value is None or value == "":
        return "NULL"
    return str(float(value))


def create_program(applicant):
    university = applicant.get("University")
    program_name = applicant.get("Program Name")

    if university is None and program_name is None:
        return "NULL"

    if university is None:
        full_program = program_name
    elif program_name is None:
        full_program = university
    else:
        full_program = program_name + ", " + university

    return format_text(full_program)


def insert_applicants(connection, applicant_data):
    for applicant in applicant_data:
        insert_query = f"""
        INSERT INTO applicants (
            program,
            comments,
            date_added,
            url,
            status,
            term,
            us_or_international,
            gpa,
            gre,
            gre_v,
            gre_aw,
            degree,
            llm_generated_program,
            llm_generated_university
        )
        VALUES (
            {create_program(applicant)},
            {format_text(applicant.get("Comments"))},
            {format_text(applicant.get("Date of Information Added to Grad Cafe"))},
            {format_text(applicant.get("URL link to applicant entry"))},
            {format_text(applicant.get("Applicant Status"))},
            {format_text(applicant.get("Semester and Year of Program Start"))},
            {format_text(applicant.get("International / American Student"))},
            {format_float(applicant.get("GPA"))},
            {format_float(applicant.get("GRE Score"))},
            {format_float(applicant.get("GRE V Score"))},
            {format_float(applicant.get("GRE AW"))},
            {format_text(applicant.get("Masters or PhD"))},
            {format_text(applicant.get("llm-generated-program"))},
            {format_text(applicant.get("llm-generated-university"))}
        );
        """

        execute_query(connection, insert_query)


insert_applicants(connection, applicant_data)

Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed successfully
Query executed succe

#### Read query

In [ ]:
# Note: the two functions are different: 
# execute_query ---- runs query, does not fetch data
# execute_read_query ---- runs SELECT query, fetches and returns data

def execute_read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except OperationalError as e:
        print(f"The error '{e}' occurred")


select_applicants = "SELECT * FROM applicants limit 10"
applicants = execute_read_query(connection, select_applicants)

for applicant in applicants:
    print(applicant)

(1, 'Bioinformatics Masters, San Jose State University', None, datetime.date(2026, 5, 31), 'https://www.thegradcafe.com/result/1020289', 'Rejected', 'Fall 2026', 'American', 3.2, None, None, None, 'Masters', 'Bioinformatics Masters', 'San Jose State University')
(2, 'Environmental Economics PhD, Stockholm University', None, datetime.date(2026, 5, 31), 'https://www.thegradcafe.com/result/1020288', 'Accepted', 'Fall 2026', 'International', None, None, None, None, 'PhD', 'Environmental Economics PhD', 'Stockholm University')
(3, 'Global Health PhD, Meharry Medical College', None, datetime.date(2026, 5, 30), 'https://www.thegradcafe.com/result/1020287', 'Accepted', 'Fall 2026', 'American', 3.9, None, None, None, 'PhD', 'Global Health PhD', 'Meharry Medical College')
(4, 'Public Policy PhD, University of Massachusetts', None, datetime.date(2026, 5, 29), 'https://www.thegradcafe.com/result/1020286', 'Waitlisted', 'Fall 2026', 'International', None, None, None, None, 'PhD', 'Public Policy PhD

In [31]:
total_applicants = "SELECT count(*) FROM applicants"
tot = execute_read_query(connection, total_applicants)
print(tot)

[(41000,)]


#### Q1: How many entries do you have in your database who have applied for Fall 2026?

In [ ]:
# first check why type of data in term
query1a = "SELECT distinct term FROM applicants"
query1a_output = execute_read_query(connection, query1a)
print(query1a_output)

[('Spring 2025',), ('Fall 2025',), ('Fall 2024',), ('Spring 2026',), ('Fall 2026',)]


In [33]:
query1 = "SELECT count(*) FROM applicants where term = 'Fall 2026'"
query1_output = execute_read_query(connection, query1)
print(query1_output)

[(33053,)]


#### Q2: What percentage of entries are from international students (not American or Other) (to two decimal places)?

In [34]:
# use triple quoted string for nice layout query!
query2 = """
SELECT
    ROUND(100.0 * SUM(CASE WHEN us_or_international = 'International' THEN 1
                           ELSE 0
                      END
                      ) / COUNT(*),
        2
    ) AS international_pct
FROM applicants;
"""

query2_output = execute_read_query(connection, query2)
print(query2_output)

[(Decimal('46.42'),)]


In [37]:
# double check query2 results
query2_check = """
SELECT distinct us_or_international, count(*)
from applicants
group by us_or_international;
"""

query2_chk_output = execute_read_query(connection, query2_check)
print(query2_chk_output)



[('American', 21220), ('International', 19032), ('Other', 96), (None, 652)]


#### Q3: What is the average GPA, GRE, GRE V, GRE AW of applicants who provide these metrics?

In [39]:
# query3 = """
# SELECT
#     ROUND(AVG(gpa), 2) AS avg_gpa,
#     ROUND(AVG(gre), 2) AS avg_gre,
#     ROUND(AVG(gre_v),2) AS avg_gre_v,
#     ROUND(AVG(gre_aw),2) AS avg_gre_aw
# FROM applicants;
# """
query3 = """
SELECT
    ROUND(AVG(gpa)::numeric, 2) AS avg_gpa,
    ROUND(AVG(gre)::numeric, 2) AS avg_gre,
    ROUND(AVG(gre_v)::numeric, 2) AS avg_gre_v,
    ROUND(AVG(gre_aw)::numeric, 2) AS avg_gre_aw
FROM applicants;
"""

query3_output = execute_read_query(connection, query3)
print(query3_output)

[(Decimal('3.76'), Decimal('264.71'), Decimal('161.34'), Decimal('4.32'))]


In [41]:

avg_gpa, avg_gre, avg_gre_v, avg_gre_aw = query3_output[0]

print("Average GPA:", avg_gpa)
print("Average GRE:", avg_gre)
print("Average GRE V:", avg_gre_v)
print("Average GRE AW:", avg_gre_aw)

Average GPA: 3.76
Average GRE: 264.71
Average GRE V: 161.34
Average GRE AW: 4.32


#### Q4: What is their average GPA of American students in Fall 2026?

In [ ]:
query4 = """
SELECT
    ROUND(AVG(gpa)::numeric, 2) AS avg_gpa
FROM applicants
where us_or_international = 'American' and term = 'Fall 2026';
"""

query4_output = execute_read_query(connection, query4)
print(query4_output)

[(Decimal('3.79'),)]


#### Q5: What percent of entries for Fall 2026 are Acceptances (to two decimal places)?

In [43]:
query5a = """
SELECT
   distinct status
FROM applicants
where term = 'Fall 2026';
"""

query5a_output = execute_read_query(connection, query5a)
print(query5a_output)

[('Interview',), ('Rejected',), ('Accepted',), ('Waitlisted',)]


In [44]:
query5 = """
SELECT
    ROUND(
        100.0 * SUM(
            CASE WHEN status = 'Accepted' THEN 1
                 ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS fall_2026_accept_pct
FROM applicants
WHERE term = 'Fall 2026';
"""

query5_output = execute_read_query(connection, query5)
print(query5_output)

[(Decimal('35.84'),)]
